# ICT playbook analyst
Drop chart screenshots in a Google Drive folder, press **Run**, and get a trade card: **direction, entry, stop loss, take profit** - or "no trade" and why.

**How it works:** candle data is downloaded automatically (Yahoo Finance); Python computes the levels, sessions and patterns; Claude reads that, your screenshots, any economic data you paste and your playbook (a Google Doc); Python then checks and sizes the idea against your risk rules.

**Drive folders (created for you):** `MyDrive/ict/screenshots` (your chart images), `MyDrive/ict/data` (optional CSV exports - used instead of Yahoo if present), `MyDrive/ict/ideas` (saved results), `MyDrive/ict/journal.json` (closed trades).

*Educational analysis only - not financial advice. It never places orders.*

## 1. Setup (once per session)

In [ ]:
import os, sys
if not os.path.isdir("/content/Coding-Projects-"):
    !git clone -q https://github.com/EliseoRiddick/Coding-Projects-.git
else:
    !git -C /content/Coding-Projects- pull -q
!pip install -q anthropic pillow yfinance
sys.path.insert(0, "/content/Coding-Projects-/ict-playbook/agent")

from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")   # left sidebar > key icon > add secret, enable notebook access
print("Ready.")

## 2. Settings and playbook

In [ ]:
#@title Settings { run: "auto" }
SYMBOL = "NQ"                      #@param {type:"string"}
EQUITY = 50000                     #@param {type:"number"}
RISK_PCT = 1.0                     #@param {type:"number"}
VALUE_PER_POINT = 0                #@param {type:"number"}
SIZE_STEP = 1                      #@param {type:"number"}
MODEL = "claude-opus-5-5"          #@param ["claude-opus-5-5", "claude-sonnet-5-5"] {allow-input: true}
EFFORT = "high"                    #@param ["low", "medium", "high", "xhigh", "max"]
ASOF = ""                          #@param {type:"string"}
PLAYBOOK_DOC_ID = "1_Lb_UD-FKp7NTl6yqI9zD9g3csx0Imj9PPV99NlOqKQ"  #@param {type:"string"}
TZ = "America/New_York"            #@param {type:"string"}
# SYMBOL: NQ, MNQ, ES, MES, YM, RTY, GC, CL (or any Yahoo ticker). VALUE_PER_POINT: 0 = use the standard value for the symbol
#   (NQ 20, MNQ 2, ES 50, MES 5). RISK_PCT: R93 default 1.0, 0.5 while learning, 2.0 is the hard ceiling.
# SIZE_STEP: smallest tradable size (1 contract). ASOF: blank = analyse the latest completed candle; or e.g. 2025-03-12 08:45
#   (New York time) to replay a past moment - Yahoo only keeps about 60 days of 5m/15m data.

import io, re
from pathlib import Path
from google.colab import auth, drive as gdrive
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

gdrive.mount("/content/drive")
ICT = Path("/content/drive/MyDrive/ict")
SHOTS_DIR, DATA_DIR, OUT_DIR, JOURNAL = ICT / "screenshots", ICT / "data", ICT / "ideas", ICT / "journal.json"
for d in (SHOTS_DIR, DATA_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

from ict_agent.market_data import point_value
if not VALUE_PER_POINT:
    VALUE_PER_POINT = point_value(SYMBOL)
    assert VALUE_PER_POINT, f"No standard point value for {SYMBOL}: set VALUE_PER_POINT yourself (value of a 1.0 price move per contract)"

auth.authenticate_user()
drive = build("drive", "v3", cache_discovery=False)
meta = drive.files().get(fileId=PLAYBOOK_DOC_ID, fields="name,mimeType,modifiedTime").execute()
if meta["mimeType"] == "application/vnd.google-apps.document":
    try:
        data = drive.files().export(fileId=PLAYBOOK_DOC_ID, mimeType="text/markdown").execute()
    except Exception:
        data = drive.files().export(fileId=PLAYBOOK_DOC_ID, mimeType="text/plain").execute()
else:                                                   # an uploaded .md/.txt file
    buf = io.BytesIO(); dl = MediaIoBaseDownload(buf, drive.files().get_media(fileId=PLAYBOOK_DOC_ID))
    done = False
    while not done: _, done = dl.next_chunk()
    data = buf.getvalue()
PLAYBOOK = data.decode("utf-8") if isinstance(data, bytes) else data
n_rules = len(set(re.findall(r"\bR\d{1,3}\b", PLAYBOOK)))
print(f"Playbook '{meta['name']}' (edited {meta['modifiedTime'][:16]}): {len(PLAYBOOK):,} chars, {n_rules} distinct rule numbers")
if n_rules < 100:
    print("WARNING: expected about 122 rules. The export may be missing tabs/sections - check the doc before trusting results.")
print(f"Point value {VALUE_PER_POINT:g}; risk {RISK_PCT}% of {EQUITY:,} = {EQUITY * RISK_PCT / 100:,.0f} per trade")
print(f"Put chart screenshots in {SHOTS_DIR}")

## 3. Economic data (optional)
Paste calendar events, rate expectations, notes. Include times (New York) and impact, e.g. `Thu 08:30 CPI m/m exp 0.3%, high impact`. Holiday weeks, FOMC and NFP matter to the playbook (R65-R67). Leave blank if none.

In [ ]:
#@title Economic data { run: "auto" }
ECON = """
""" #@param {type:"string"}
ECON = ECON.strip()
print(f"{len(ECON)} characters of economic data" if ECON else "No economic data")

## 4. Run
Tick **DRY_RUN** to preview exactly what would be sent, without calling Claude (free). The newest screenshots in the folder are used (up to `MAX_SHOTS`).

In [ ]:
DRY_RUN = False   #@param {type:"boolean"}
MAX_SHOTS = 6     #@param {type:"integer"}

import pandas as pd
from IPython.display import Markdown, display
from ict_agent.analyze import prepare, run_analysis
from ict_agent.data import load_timeframes
from ict_agent.images import load_folder
from ict_agent.market_data import fetch_yahoo
from ict_agent.risk import RiskConfig
from ict_agent.trade_card import card_markdown, make_card, report_markdown

# 1. candles: your CSVs if present, otherwise Yahoo
if list(DATA_DIR.glob("*.csv")):
    frames = load_timeframes(DATA_DIR, TZ, derive=["1h", "4h", "1d"])
    print(f"Using CSV files from {DATA_DIR}: {list(frames)}")
else:
    market = fetch_yahoo(SYMBOL, TZ)
    frames = market.frames
    print(f"Downloaded {market.ticker} candles from Yahoo: {list(frames)}; newest completed candle ended "
          f"{market.last_bar_end:%Y-%m-%d %H:%M} ({market.age_minutes:.0f} min ago)")
    for w in market.warnings:
        print(" -", w)
for tf, df in frames.items():
    print(f"  {tf:>3}: {len(df):>6} bars, last {df.index[-1]:%Y-%m-%d %H:%M}, close {df['close'].iloc[-1]:.2f}")

# 2. screenshots
IMAGES, SHOT_NAMES, skipped = load_folder(SHOTS_DIR, MAX_SHOTS)
print(f"{len(IMAGES)} screenshot(s): {SHOT_NAMES}" + (f"  (skipped unreadable: {skipped})" if skipped else ""))
if not IMAGES:
    print("No screenshots found - the analysis will use the candle data only.")

# 3. analyse
asof = pd.Timestamp(ASOF, tz=TZ) if ASOF.strip() else None
RISK = RiskConfig(EQUITY, VALUE_PER_POINT, RISK_PCT, SIZE_STEP)   # raises if RISK_PCT is above the 2% ceiling
if DRY_RUN:
    features, system, content = prepare(frames, SYMBOL, PLAYBOOK, ECON, IMAGES, asof, SHOT_NAMES)
    print(content[-1]["text"][:6000])
else:
    result, features = run_analysis(frames, SYMBOL, PLAYBOOK, ECON, IMAGES, asof, MODEL, EFFORT, OUT_DIR, RISK, JOURNAL, SHOT_NAMES)
    card = make_card(result, features)
    display(Markdown(card_markdown(card)))
    display(Markdown("---\n" + report_markdown(result, features, str(OUT_DIR))))

## 5. Log a closed trade (keeps R95/R96 working)
After a trade closes, record its result so the next run halves or restores risk correctly. `RISK_AMOUNT` is what you planned to risk on it; `PNL` is negative for a loss.

In [ ]:
#@title Log a closed trade { run: "auto" }
PNL = 0.0           #@param {type:"number"}
RISK_AMOUNT = 0.0   #@param {type:"number"}
LOG_IT = False      #@param {type:"boolean"}

from ict_agent.risk import log_trade, load_journal, risk_state, RiskConfig
if LOG_IT and RISK_AMOUNT > 0:
    log_trade(JOURNAL, PNL, RISK_AMOUNT, SYMBOL, RISK_PCT)
    print("Logged. Untick LOG_IT so it is not logged twice.")
state = risk_state(load_journal(JOURNAL), RiskConfig(EQUITY, VALUE_PER_POINT, RISK_PCT, SIZE_STEP))
print(f"{state['trades_in_journal']} trades in journal -> next trade risk {state['effective_risk_pct']}%")
for r in state["reasons"]: print(" -", r)